In [1]:
import os
import pandas as pd
from collections import Counter
from itertools import islice

In [2]:
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)

assignment_1_path = "../Assignment_1/tokenized_output"

print(os.listdir(assignment_1_path))

['hin_corpus_tokenized.parquet']


In [3]:
files = [
    os.path.join(assignment_1_path, f)
    for f in os.listdir(assignment_1_path)
    if f.endswith(".parquet")
]

files

['../Assignment_1/tokenized_output\\hin_corpus_tokenized.parquet']

In [4]:
dfs = []

for file in files:
    df_temp = pd.read_parquet(file)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

print("Total sentences available:", len(df))

Total sentences available: 1496811


In [5]:
df = df.iloc[:1000000].copy()

df = df[["sentence", "tokens"]].reset_index(drop=True)

print(len(df))
df.head()


1000000


,sentence,tokens
0,लोगों को बिलों संबंधी सुविधा देना ही उनका काम\...,"[लोगों, को, बिलों, संबंधी, सुविधा, देना, ही, उ..."
1,हालांकि तब पार्टी पर देवीलाल की मजबूत पकड़ के ...,"[हालांकि, तब, पार्टी, पर, देवीलाल, की, मजबूत, ..."
2,1989 में देवीलाल केन्द्र की राजनीति में सक्रिय...,"[1989, में, देवीलाल, केन्द्र, की, राजनीति, में..."
3,उन परिस्थितियों में देवीलाल ने कड़ा निर्णय लेत...,"[उन, परिस्थितियों, में, देवीलाल, ने, कड़ा, निर..."
4,उस समय रणजीत की नाराजगी के चलते उनके समर्थन मे...,"[उस, समय, रणजीत, की, नाराजगी, के, चलते, उनके, ..."


In [6]:
train_df = df.iloc[:998000].copy()
dev_df = df.iloc[998000:999000].copy()
test_df = df.iloc[999000:1000000].copy()

print("Training sentences:", len(train_df))
print("Development sentences:", len(dev_df))
print("Test sentences:", len(test_df))

Training sentences: 998000
Development sentences: 1000
Test sentences: 1000


In [7]:
train_df.to_parquet(
    "data/train.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy"
)

dev_df.to_parquet(
    "data/dev.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy"
)

test_df.to_parquet(
    "data/test.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy"
)

In [8]:
def prepare_sentence(tokens):
    return ["<s>"] + list(tokens) + ["</s>"]

In [9]:
train_sentences = [
    prepare_sentence(tokens)
    for tokens in train_df["tokens"]
]

dev_sentences = [
    prepare_sentence(tokens)
    for tokens in dev_df["tokens"]
]

test_sentences = [
    prepare_sentence(tokens)
    for tokens in test_df["tokens"]
]

print(train_sentences[0])

['<s>', 'लोगों', 'को', 'बिलों', 'संबंधी', 'सुविधा', 'देना', 'ही', 'उनका', 'काम', 'इनेलो', '1987', 'में', 'उस', 'वक्त', 'ऐसे', 'ही', 'दोराहे', 'पर', 'खड़ी', 'थी', ',', 'जब', 'पूर्व', 'उपप्रधानमंत्री', 'देवीलाल', 'ने', 'अपने', 'पुत्र', 'ओमप्रकाश', 'चौटाला', 'को', 'अपना', 'राजनीतिक', 'उत्तराधिकारी', 'घोषित', 'किया', 'था', '।', '</s>']


In [10]:
def build_ngrams(sentences, n):
    counts = Counter()

    for sentence in sentences:
        for i in range(len(sentence) - n + 1):
            ngram = tuple(sentence[i:i+n])
            counts[ngram] += 1

    return counts

In [11]:
unigram_counts = build_ngrams(train_sentences, 1)
bigram_counts = build_ngrams(train_sentences, 2)
trigram_counts = build_ngrams(train_sentences, 3)
quadrigram_counts = build_ngrams(train_sentences, 4)

In [12]:
print("Unigram types:", len(unigram_counts))
print("Bigram types:", len(bigram_counts))
print("Trigram types:", len(trigram_counts))
print("Quadrigram types:", len(quadrigram_counts))

Unigram types: 307861
Bigram types: 3782573
Trigram types: 10720662
Quadrigram types: 15632208


In [13]:
vocabulary = set(unigram_counts.keys())

vocabulary_size = len(vocabulary)

print("Vocabulary size:", vocabulary_size)

Vocabulary size: 307861


In [14]:
total_unigrams = sum(unigram_counts.values())

unigram_probabilities = {
    ngram: count / total_unigrams
    for ngram, count in unigram_counts.items()
}

print(list(unigram_probabilities.items())[:10])

[(('<s>',), 0.04144558547620438), (('लोगों',), 0.0017228157449702792), (('को',), 0.01607714958691874), (('बिलों',), 7.724327553681377e-06), (('संबंधी',), 7.674493182367304e-05), (('सुविधा',), 0.00013405445883485744), (('देना',), 0.0001627922796259731), (('ही',), 0.003947214437216707), (('उनका',), 0.00039497892130679343), (('काम',), 0.0009110553649067853)]


In [15]:
context_counts_bigram = Counter()

for (w1, w2), count in bigram_counts.items():
    context_counts_bigram[(w1,)] += count

bigram_probabilities = {
    (w1, w2): count / context_counts_bigram[(w1,)]
    for (w1, w2), count in bigram_counts.items()
}

In [16]:
context_counts_trigram = Counter()

for (w1, w2, w3), count in trigram_counts.items():
    context_counts_trigram[(w1, w2)] += count

trigram_probabilities = {
    (w1, w2, w3): count / context_counts_trigram[(w1, w2)]
    for (w1, w2, w3), count in trigram_counts.items()
}

In [17]:
context_counts_quadrigram = Counter()

for (w1, w2, w3, w4), count in quadrigram_counts.items():
    context_counts_quadrigram[(w1, w2, w3)] += count

quadrigram_probabilities = {
    (w1, w2, w3, w4): count / context_counts_quadrigram[(w1, w2, w3)]
    for (w1, w2, w3, w4), count in quadrigram_counts.items()
}

In [18]:
import pickle

In [19]:
with open("models/unigram_counts.pkl", "wb") as f:
    pickle.dump(unigram_counts, f)

with open("models/bigram_counts.pkl", "wb") as f:
    pickle.dump(bigram_counts, f)

with open("models/trigram_counts.pkl", "wb") as f:
    pickle.dump(trigram_counts, f)

with open("models/quadrigram_counts.pkl", "wb") as f:
    pickle.dump(quadrigram_counts, f)

In [20]:
with open("models/unigram_probabilities.pkl", "wb") as f:
    pickle.dump(unigram_probabilities, f)

with open("models/bigram_probabilities.pkl", "wb") as f:
    pickle.dump(bigram_probabilities, f)

with open("models/trigram_probabilities.pkl", "wb") as f:
    pickle.dump(trigram_probabilities, f)

with open("models/quadrigram_probabilities.pkl", "wb") as f:
    pickle.dump(quadrigram_probabilities, f)

In [21]:
with open("models/vocabulary.pkl", "wb") as f:
    pickle.dump(vocabulary, f)

In [22]:
print("Total unigram tokens:", sum(unigram_counts.values()))
print("Unique unigrams:", len(unigram_counts))
print("Unique bigrams:", len(bigram_counts))
print("Unique trigrams:", len(trigram_counts))
print("Unique quadrigrams:", len(quadrigram_counts))

Total unigram tokens: 24079766
Unique unigrams: 307861
Unique bigrams: 3782573
Unique trigrams: 10720662
Unique quadrigrams: 15632208


In [23]:
print("Sample unigram probabilities:")

for ngram, probability in list(unigram_probabilities.items())[:10]:
    print(ngram, probability)

Sample unigram probabilities:
('<s>',) 0.04144558547620438
('लोगों',) 0.0017228157449702792
('को',) 0.01607714958691874
('बिलों',) 7.724327553681377e-06
('संबंधी',) 7.674493182367304e-05
('सुविधा',) 0.00013405445883485744
('देना',) 0.0001627922796259731
('ही',) 0.003947214437216707
('उनका',) 0.00039497892130679343
('काम',) 0.0009110553649067853


In [24]:
print("Sample bigram probabilities:")

for ngram, probability in list(bigram_probabilities.items())[:10]:
    print(ngram, probability)

Sample bigram probabilities:
('<s>', 'लोगों') 0.0016042084168336674
('लोगों', 'को') 0.3115824996986863
('को', 'बिलों') 7.749254779998656e-06
('बिलों', 'संबंधी') 0.005376344086021506
('संबंधी', 'सुविधा') 0.0005411255411255411
('सुविधा', 'देना') 0.0021685254027261464
('देना', 'ही') 0.009438775510204082
('ही', 'उनका') 0.0015360659877114722
('उनका', 'काम') 0.007044474818631059
('काम', 'इनेलो') 4.5583006655118975e-05


In [25]:
print("Sample trigram probabilities:")

for ngram, probability in list(trigram_probabilities.items())[:10]:
    print(ngram, probability)

Sample trigram probabilities:
('<s>', 'लोगों', 'को') 0.2773266708307308
('लोगों', 'को', 'बिलों') 7.736345350456444e-05
('को', 'बिलों', 'संबंधी') 0.3333333333333333
('बिलों', 'संबंधी', 'सुविधा') 1.0
('संबंधी', 'सुविधा', 'देना') 1.0
('सुविधा', 'देना', 'ही') 0.14285714285714285
('देना', 'ही', 'उनका') 0.02702702702702703
('ही', 'उनका', 'काम') 0.0136986301369863
('उनका', 'काम', 'इनेलो') 0.014925373134328358
('काम', 'इनेलो', '1987') 1.0


In [26]:
print("Sample quadrigram probabilities:")

for ngram, probability in list(quadrigram_probabilities.items())[:10]:
    print(ngram, probability)

Sample quadrigram probabilities:
('<s>', 'लोगों', 'को', 'बिलों') 0.0022522522522522522
('लोगों', 'को', 'बिलों', 'संबंधी') 1.0
('को', 'बिलों', 'संबंधी', 'सुविधा') 1.0
('बिलों', 'संबंधी', 'सुविधा', 'देना') 1.0
('संबंधी', 'सुविधा', 'देना', 'ही') 1.0
('सुविधा', 'देना', 'ही', 'उनका') 1.0
('देना', 'ही', 'उनका', 'काम') 1.0
('ही', 'उनका', 'काम', 'इनेलो') 0.5
('उनका', 'काम', 'इनेलो', '1987') 1.0
('काम', 'इनेलो', '1987', 'में') 1.0


In [27]:
summary = pd.DataFrame({
    "Model": [
        "Unigram",
        "Bigram",
        "Trigram",
        "Quadrigram"
    ],
    "N-gram Types": [
        len(unigram_counts),
        len(bigram_counts),
        len(trigram_counts),
        len(quadrigram_counts)
    ]
})

summary

,Model,N-gram Types
0,Unigram,307861
1,Bigram,3782573
2,Trigram,10720662
3,Quadrigram,15632208


In [28]:
print("Data split completed successfully")
print()
print(f"Training set:     {len(train_df):,} sentences")
print(f"Development set:  {len(dev_df):,} sentences")
print(f"Test set:         {len(test_df):,} sentences")
print(f"Total:            {len(df):,} sentences")

Data split completed successfully

Training set:     998,000 sentences
Development set:  1,000 sentences
Test set:         1,000 sentences
Total:            1,000,000 sentences


## Question 2

In [29]:
import pickle
import os
from collections import Counter

In [30]:
with open("models/unigram_counts.pkl", "rb") as f:
    unigram_counts = pickle.load(f)

with open("models/bigram_counts.pkl", "rb") as f:
    bigram_counts = pickle.load(f)

with open("models/trigram_counts.pkl", "rb") as f:
    trigram_counts = pickle.load(f)

with open("models/quadrigram_counts.pkl", "rb") as f:
    quadrigram_counts = pickle.load(f)

with open("models/vocabulary.pkl", "rb") as f:
    vocabulary = pickle.load(f)

In [31]:
V = len(vocabulary)

print(f"Vocabulary size (V): {V:,}")

Vocabulary size (V): 307,861


In [32]:
total_unigram_tokens = sum(unigram_counts.values())

print(f"Total unigram tokens: {total_unigram_tokens:,}")

Total unigram tokens: 24,079,766


In [33]:
unigram_laplace_probabilities = {}

denominator = total_unigram_tokens + V

for word in vocabulary:
    count = unigram_counts.get((word,), 0)
    unigram_laplace_probabilities[word] = (count + 1) / denominator

In [34]:
print("Sample Laplace-smoothed unigram probabilities:")

for word, probability in list(unigram_laplace_probabilities.items())[:20]:
    print(word, probability)

Sample Laplace-smoothed unigram probabilities:
('परीक्षण',) 4.100439948503395e-08
('कंवर्जेस',) 4.100439948503395e-08
('36.75',) 4.100439948503395e-08
('विनोथ',) 4.100439948503395e-08
('नृत्या',) 4.100439948503395e-08
('मान्टू',) 4.100439948503395e-08
('गलियाकोट',) 4.100439948503395e-08
('कडी',) 4.100439948503395e-08
('मोजूद',) 4.100439948503395e-08
('मॉनीटर्स',) 4.100439948503395e-08
('निनटेन्डो',) 4.100439948503395e-08
('पुरूषविहीन',) 4.100439948503395e-08
('यूनिटॉप',) 4.100439948503395e-08
('ट्रैपर',) 4.100439948503395e-08
('ठोंकने',) 4.100439948503395e-08
('12279',) 4.100439948503395e-08
('थामा',) 4.100439948503395e-08
('गर्तरोमों',) 4.100439948503395e-08
('निर्देशो',) 4.100439948503395e-08
('सेक्रेटियों',) 4.100439948503395e-08


In [35]:
bigram_context_counts = Counter()

for (w1, w2), count in bigram_counts.items():
    bigram_context_counts[w1] += count

In [36]:
bigram_laplace_probabilities = {}

for (w1, w2), count in bigram_counts.items():
    numerator = count + 1
    denominator = bigram_context_counts[w1] + V
    bigram_laplace_probabilities[(w1, w2)] = numerator / denominator

In [37]:
print("Sample Laplace-smoothed bigram probabilities:")

for ngram, probability in list(bigram_laplace_probabilities.items())[:20]:
    print(ngram, probability)

Sample Laplace-smoothed bigram probabilities:
('<s>', 'लोगों') 0.0012267768162155083
('लोगों', 'को') 0.0370034292649694
('को', 'बिलों') 5.7554370894754635e-06
('बिलों', 'संबंधी') 6.492515752466345e-06
('संबंधी', 'सुविधा') 6.4576747850401506e-06
('सुविधा', 'देना') 2.5716113395202017e-05
('देना', 'ही') 0.00012188042247603285
('ही', 'उनका') 0.0003648466527181076
('उनका', 'काम') 0.0002142596070226737
('काम', 'इनेलो') 6.064299770466254e-06
('इनेलो', '1987') 6.494286651318503e-06
('1987', 'में') 0.0002435041217131002
('में', 'उस') 0.0003157485153441021
('उस', 'वक्त') 0.0025962623855512353
('वक्त', 'ऐसे') 1.2826844019163305e-05
('ऐसे', 'ही') 0.0025634935079758272
('ही', 'दोराहे') 4.963900036981055e-06
('दोराहे', 'पर') 3.2481136579931206e-05
('पर', 'खड़ी') 0.00015360272413505937
('खड़ी', 'थी') 0.00014252535493672198


In [38]:
trigram_context_counts = Counter()

for (w1, w2, w3), count in trigram_counts.items():
    trigram_context_counts[(w1, w2)] += count

In [39]:
trigram_laplace_probabilities = {}

for (w1, w2, w3), count in trigram_counts.items():
    numerator = count + 1
    denominator = trigram_context_counts[(w1, w2)] + V
    trigram_laplace_probabilities[(w1, w2, w3)] = numerator / denominator

In [40]:
print("Sample Laplace-smoothed trigram probabilities:")

for ngram, probability in list(trigram_laplace_probabilities.items())[:20]:
    print(ngram, probability)

Sample Laplace-smoothed trigram probabilities:
('<s>', 'लोगों', 'को') 0.0014379794611293147
('लोगों', 'को', 'बिलों') 6.234666616789334e-06
('को', 'बिलों', 'संबंधी') 6.496375022737313e-06
('बिलों', 'संबंधी', 'सुविधा') 6.496417225899916e-06
('संबंधी', 'सुविधा', 'देना') 6.496417225899916e-06
('सुविधा', 'देना', 'ही') 6.49629061805709e-06
('देना', 'ही', 'उनका') 6.495657652859064e-06
('ही', 'उनका', 'काम') 9.7400383757512e-06
('उनका', 'काम', 'इनेलो') 6.495024810994778e-06
('काम', 'इनेलो', '1987') 6.496417225899916e-06
('इनेलो', '1987', 'में') 6.496417225899916e-06
('1987', 'में', 'उस') 6.4948771656356046e-06
('में', 'उस', 'वक्त') 0.00012980396356402743
('उस', 'वक्त', 'ऐसे') 6.479033846472814e-06
('वक्त', 'ऐसे', 'ही') 6.496375022737313e-06
('ऐसे', 'ही', 'दोराहे') 6.478970880265378e-06
('ही', 'दोराहे', 'पर') 6.496417225899916e-06
('दोराहे', 'पर', 'खड़ी') 6.496248416539448e-06
('पर', 'खड़ी', 'थी') 2.597883374520609e-05
('खड़ी', 'थी', ',') 1.9486593223861983e-05


In [41]:
quadrigram_context_counts = Counter()

for (w1, w2, w3, w4), count in quadrigram_counts.items():
    quadrigram_context_counts[(w1, w2, w3)] += count

In [42]:
quadrigram_laplace_probabilities = {}

for (w1, w2, w3, w4), count in quadrigram_counts.items():
    numerator = count + 1
    denominator = quadrigram_context_counts[(w1, w2, w3)] + V
    quadrigram_laplace_probabilities[(w1, w2, w3, w4)] = numerator / denominator

In [43]:
print("Sample Laplace-smoothed quadrigram probabilities:")

for ngram, probability in list(quadrigram_laplace_probabilities.items())[:20]:
    print(ngram, probability)

Sample Laplace-smoothed quadrigram probabilities:
('<s>', 'लोगों', 'को', 'बिलों') 6.487082596779164e-06
('लोगों', 'को', 'बिलों', 'संबंधी') 6.496417225899916e-06
('को', 'बिलों', 'संबंधी', 'सुविधा') 6.496417225899916e-06
('बिलों', 'संबंधी', 'सुविधा', 'देना') 6.496417225899916e-06
('संबंधी', 'सुविधा', 'देना', 'ही') 6.496417225899916e-06
('सुविधा', 'देना', 'ही', 'उनका') 6.496417225899916e-06
('देना', 'ही', 'उनका', 'काम') 6.496417225899916e-06
('ही', 'उनका', 'काम', 'इनेलो') 6.496396124250072e-06
('उनका', 'काम', 'इनेलो', '1987') 6.496417225899916e-06
('काम', 'इनेलो', '1987', 'में') 6.496417225899916e-06
('इनेलो', '1987', 'में', 'उस') 6.496417225899916e-06
('1987', 'में', 'उस', 'वक्त') 6.496417225899916e-06
('में', 'उस', 'वक्त', 'ऐसे') 6.4956154595647935e-06
('उस', 'वक्त', 'ऐसे', 'ही') 6.496417225899916e-06
('वक्त', 'ऐसे', 'ही', 'दोराहे') 6.496417225899916e-06
('ऐसे', 'ही', 'दोराहे', 'पर') 6.496417225899916e-06
('ही', 'दोराहे', 'पर', 'खड़ी') 6.496417225899916e-06
('दोराहे', 'पर', 'खड़ी', 'थी'

In [44]:
def laplace_unigram(word):
    count = unigram_counts.get((word,), 0)
    return (count + 1) / (total_unigram_tokens + V)


def laplace_bigram(w1, w2):
    numerator = bigram_counts.get((w1, w2), 0) + 1
    denominator = bigram_context_counts.get(w1, 0) + V
    return numerator / denominator


def laplace_trigram(w1, w2, w3):
    numerator = trigram_counts.get((w1, w2, w3), 0) + 1
    denominator = trigram_context_counts.get((w1, w2), 0) + V
    return numerator / denominator


def laplace_quadrigram(w1, w2, w3, w4):
    numerator = quadrigram_counts.get((w1, w2, w3, w4), 0) + 1
    denominator = quadrigram_context_counts.get((w1, w2, w3), 0) + V
    return numerator / denominator

In [45]:
print("Unigram:", laplace_unigram("भारत"))

print(
    "Bigram:",
    laplace_bigram("भारत", "में")
)

print(
    "Trigram:",
    laplace_trigram("भारत", "में", "लोग")
)

print(
    "Quadrigram:",
    laplace_quadrigram("भारत", "में", "लोग", "रहते")
)

Unigram: 0.00114152147726386
Bigram: 0.015275589143846123
Trigram: 4.473014939869899e-05
Quadrigram: 3.248082007574527e-06


In [46]:
print("Unseen unigram:", laplace_unigram("xyzabc"))

print(
    "Unseen bigram:",
    laplace_bigram("xyzabc", "qwerty")
)

print(
    "Unseen trigram:",
    laplace_trigram("xyzabc", "qwerty", "asdfgh")
)

print(
    "Unseen quadrigram:",
    laplace_quadrigram("xyzabc", "qwerty", "asdfgh", "zxcvbn")
)

Unseen unigram: 4.100439948503395e-08
Unseen bigram: 3.2482191638434227e-06
Unseen trigram: 3.2482191638434227e-06
Unseen quadrigram: 3.2482191638434227e-06


In [47]:
with open("models/unigram_laplace.pkl", "wb") as f:
    pickle.dump(unigram_laplace_probabilities, f)

with open("models/bigram_laplace.pkl", "wb") as f:
    pickle.dump(bigram_laplace_probabilities, f)

with open("models/trigram_laplace.pkl", "wb") as f:
    pickle.dump(trigram_laplace_probabilities, f)

with open("models/quadrigram_laplace.pkl", "wb") as f:
    pickle.dump(quadrigram_laplace_probabilities, f)

In [48]:
with open("models/bigram_context_counts.pkl", "wb") as f:
    pickle.dump(bigram_context_counts, f)

with open("models/trigram_context_counts.pkl", "wb") as f:
    pickle.dump(trigram_context_counts, f)

with open("models/quadrigram_context_counts.pkl", "wb") as f:
    pickle.dump(quadrigram_context_counts, f)

In [49]:
summary = {
    "Vocabulary Size": V,
    "Total Unigram Tokens": total_unigram_tokens,
    "Unique Unigrams": len(unigram_counts),
    "Unique Bigrams": len(bigram_counts),
    "Unique Trigrams": len(trigram_counts),
    "Unique Quadrigrams": len(quadrigram_counts)
}

for key, value in summary.items():
    print(f"{key}: {value:,}")

Vocabulary Size: 307,861
Total Unigram Tokens: 24,079,766
Unique Unigrams: 307,861
Unique Bigrams: 3,782,573
Unique Trigrams: 10,720,662
Unique Quadrigrams: 15,632,208


In [50]:
print(os.listdir("models"))

['bigram_context_counts.pkl', 'bigram_counts.pkl', 'bigram_laplace.pkl', 'bigram_probabilities.pkl', 'quadrigram_context_counts.pkl', 'quadrigram_counts.pkl', 'quadrigram_laplace.pkl', 'quadrigram_probabilities.pkl', 'trigram_context_counts.pkl', 'trigram_counts.pkl', 'trigram_laplace.pkl', 'trigram_probabilities.pkl', 'unigram_counts.pkl', 'unigram_laplace.pkl', 'unigram_probabilities.pkl', 'vocabulary.pkl']


## Model Evaluation

In [51]:
import math

In [52]:
def evaluate_unigram(sentences):
    total_log_probability = 0.0
    total_tokens = 0

    for sentence in sentences:
        for word in sentence:
            probability = laplace_unigram(word)
            total_log_probability += math.log2(probability)
            total_tokens += 1

    cross_entropy = -total_log_probability / total_tokens
    perplexity = 2 ** cross_entropy

    return total_log_probability, total_tokens, cross_entropy, perplexity

In [53]:
def evaluate_bigram(sentences):
    total_log_probability = 0.0
    total_tokens = 0

    for sentence in sentences:
        for i in range(1, len(sentence)):
            w1 = sentence[i - 1]
            w2 = sentence[i]

            probability = laplace_bigram(w1, w2)

            total_log_probability += math.log2(probability)
            total_tokens += 1

    cross_entropy = -total_log_probability / total_tokens
    perplexity = 2 ** cross_entropy

    return total_log_probability, total_tokens, cross_entropy, perplexity

In [54]:
def evaluate_trigram(sentences):
    total_log_probability = 0.0
    total_tokens = 0

    for sentence in sentences:
        for i in range(2, len(sentence)):
            w1 = sentence[i - 2]
            w2 = sentence[i - 1]
            w3 = sentence[i]

            probability = laplace_trigram(w1, w2, w3)

            total_log_probability += math.log2(probability)
            total_tokens += 1

    cross_entropy = -total_log_probability / total_tokens
    perplexity = 2 ** cross_entropy

    return total_log_probability, total_tokens, cross_entropy, perplexity

In [55]:
def evaluate_quadrigram(sentences):
    total_log_probability = 0.0
    total_tokens = 0

    for sentence in sentences:
        for i in range(3, len(sentence)):
            w1 = sentence[i - 3]
            w2 = sentence[i - 2]
            w3 = sentence[i - 1]
            w4 = sentence[i]

            probability = laplace_quadrigram(
                w1,
                w2,
                w3,
                w4
            )

            total_log_probability += math.log2(probability)
            total_tokens += 1

    cross_entropy = -total_log_probability / total_tokens
    perplexity = 2 ** cross_entropy

    return total_log_probability, total_tokens, cross_entropy, perplexity

In [56]:
dev_unigram = evaluate_unigram(dev_sentences)

dev_bigram = evaluate_bigram(dev_sentences)

dev_trigram = evaluate_trigram(dev_sentences)

dev_quadrigram = evaluate_quadrigram(dev_sentences)

In [57]:
dev_results = pd.DataFrame({
    "Model": [
        "Unigram",
        "Bigram",
        "Trigram",
        "Quadrigram"
    ],
    "Log Probability": [
        dev_unigram[0],
        dev_bigram[0],
        dev_trigram[0],
        dev_quadrigram[0]
    ],
    "Tokens": [
        dev_unigram[1],
        dev_bigram[1],
        dev_trigram[1],
        dev_quadrigram[1]
    ],
    "Cross Entropy": [
        dev_unigram[2],
        dev_bigram[2],
        dev_trigram[2],
        dev_quadrigram[2]
    ],
    "Perplexity": [
        dev_unigram[3],
        dev_bigram[3],
        dev_trigram[3],
        dev_quadrigram[3]
    ]
})

dev_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-260023.365597,24954,10.420108,1370.140022
1,Bigram,-277750.972328,23954,11.595181,3093.836578
2,Trigram,-347276.527817,22954,15.129238,35838.880352
3,Quadrigram,-373121.776966,21954,16.995617,130674.404699


In [58]:
test_unigram = evaluate_unigram(test_sentences)

test_bigram = evaluate_bigram(test_sentences)

test_trigram = evaluate_trigram(test_sentences)

test_quadrigram = evaluate_quadrigram(test_sentences)

In [59]:
test_results = pd.DataFrame({
    "Model": [
        "Unigram",
        "Bigram",
        "Trigram",
        "Quadrigram"
    ],
    "Log Probability": [
        test_unigram[0],
        test_bigram[0],
        test_trigram[0],
        test_quadrigram[0]
    ],
    "Tokens": [
        test_unigram[1],
        test_bigram[1],
        test_trigram[1],
        test_quadrigram[1]
    ],
    "Cross Entropy": [
        test_unigram[2],
        test_bigram[2],
        test_trigram[2],
        test_quadrigram[2]
    ],
    "Perplexity": [
        test_unigram[3],
        test_bigram[3],
        test_trigram[3],
        test_quadrigram[3]
    ]
})

test_results

,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Unigram,-286362.806995,27068,10.579386,1530.073785
1,Bigram,-306175.844416,26068,11.745276,3433.051068
2,Trigram,-382042.330082,25068,15.240240,38705.199136
3,Quadrigram,-410389.190656,24068,17.051238,135810.713882


In [60]:
print("Development Set Results")
print()

for _, row in dev_results.iterrows():
    print(row["Model"])
    print(f"Log Probability: {row['Log Probability']:.4f}")
    print(f"Cross Entropy: {row['Cross Entropy']:.4f}")
    print(f"Perplexity: {row['Perplexity']:.4f}")
    print()

Development Set Results

Unigram
Log Probability: -260023.3656
Cross Entropy: 10.4201
Perplexity: 1370.1400

Bigram
Log Probability: -277750.9723
Cross Entropy: 11.5952
Perplexity: 3093.8366

Trigram
Log Probability: -347276.5278
Cross Entropy: 15.1292
Perplexity: 35838.8804

Quadrigram
Log Probability: -373121.7770
Cross Entropy: 16.9956
Perplexity: 130674.4047



In [61]:
print("Test Set Results")
print()

for _, row in test_results.iterrows():
    print(row["Model"])
    print(f"Log Probability: {row['Log Probability']:.4f}")
    print(f"Cross Entropy: {row['Cross Entropy']:.4f}")
    print(f"Perplexity: {row['Perplexity']:.4f}")
    print()

Test Set Results

Unigram
Log Probability: -286362.8070
Cross Entropy: 10.5794
Perplexity: 1530.0738

Bigram
Log Probability: -306175.8444
Cross Entropy: 11.7453
Perplexity: 3433.0511

Trigram
Log Probability: -382042.3301
Cross Entropy: 15.2402
Perplexity: 38705.1991

Quadrigram
Log Probability: -410389.1907
Cross Entropy: 17.0512
Perplexity: 135810.7139



In [62]:
os.makedirs("results", exist_ok=True)

dev_results.to_csv(
    "results/development_results.csv",
    index=False
)

test_results.to_csv(
    "results/test_results.csv",
    index=False
)

In [63]:
dev_results_final = dev_results.copy()
dev_results_final["Dataset"] = "Development"

test_results_final = test_results.copy()
test_results_final["Dataset"] = "Test"

all_results = pd.concat(
    [
        dev_results_final,
        test_results_final
    ],
    ignore_index=True
)

all_results = all_results[
    [
        "Dataset",
        "Model",
        "Log Probability",
        "Tokens",
        "Cross Entropy",
        "Perplexity"
    ]
]

all_results.to_csv(
    "results/laplace_results.csv",
    index=False
)

all_results

,Dataset,Model,Log Probability,Tokens,Cross Entropy,Perplexity
0,Development,Unigram,-260023.365597,24954,10.420108,1370.140022
1,Development,Bigram,-277750.972328,23954,11.595181,3093.836578
2,Development,Trigram,-347276.527817,22954,15.129238,35838.880352
3,Development,Quadrigram,-373121.776966,21954,16.995617,130674.404699
4,Test,Unigram,-286362.806995,27068,10.579386,1530.073785
5,Test,Bigram,-306175.844416,26068,11.745276,3433.051068
6,Test,Trigram,-382042.330082,25068,15.240240,38705.199136
7,Test,Quadrigram,-410389.190656,24068,17.051238,135810.713882


In [64]:
print("Assignment 4 evaluation completed")
print()
print("Smoothing method: Laplace (Add-1)")
print("Vocabulary size:", V)
print()
print("Development sentences:", len(dev_df))
print("Test sentences:", len(test_df))
print()
print("Models:")
print("1. Unigram")
print("2. Bigram")
print("3. Trigram")
print("4. Quadrigram")

Assignment 4 evaluation completed

Smoothing method: Laplace (Add-1)
Vocabulary size: 307861

Development sentences: 1000
Test sentences: 1000

Models:
1. Unigram
2. Bigram
3. Trigram
4. Quadrigram
